# Lecture 14.2 Детекция объектов

**Тема:** детекция объектов от постановки задачи до Faster R-CNN  
**Цель:** понять, что именно предсказывает детектор, как он оценивается, как устроен Faster R-CNN и как запускать реальный пример в PyTorch



## План

1. Что предсказывает детекция и чем она сложнее классификации  
2. Форматы bounding boxes, IoU, confidence scores и NMS  
3. One-stage и two-stage детекторы  
4. Архитектура Faster R-CNN подробнее  
5. Реальный датасет Penn-Fudan Pedestrian  
6. Практический fine-tuning и визуализация в PyTorch  
7. Оценка через AP и mAP  
8. Области применения, типичные ошибки и упражнения


## 1. Что такое детекция объектов?

Детекция объектов требует от модели одновременно решить три подзадачи:

1. понять, есть ли объект  
2. определить его класс  
3. локализовать его на изображении

Это уже сложнее классификации, потому что выход становится структурированным. Модель возвращает не одну метку, а **набор** предсказаний, где каждое предсказание содержит:

- bounding box  
- класс  
- confidence score

То есть модель должна рассуждать не только о содержании изображения, но и о геометрии, а также о множественности объектов: их может быть ноль, один или много.


![Detection pipeline](images/detection_pipeline.svg)

Схема выше отражает общую логику: детектор превращает одно изображение в набор candidate boxes, затем оценивает и фильтрует их. Фильтрация важна, потому что современные детекторы обычно предсказывают несколько перекрывающихся гипотез для одного и того же объекта.


## 2. Bounding boxes и форматы координат

Прямоугольник кажется простой вещью, пока вы не начнете переносить данные между библиотеками и датасетами. Тогда формат координат быстро становится одним из главных источников ошибок.

Три наиболее частых формата:

- `xyxy`: левый, верхний, правый, нижний край  
- `xywh`: левый верхний угол, ширина, высота  
- `cxcywh`: центр по x, центр по y, ширина, высота

Математически они эквивалентны, но неверная интерпретация дает систематические сбои:

- боксы смещаются  
- размеры становятся неправильными  
- метрики рушатся  
- визуализация выглядит «почти правильно», но не совсем

Поэтому нужно всегда сначала проверять формат координат.


## 3. IoU, confidence и NMS

### IoU

Intersection over Union показывает, насколько сильно перекрываются два бокса:

$$
IoU = \frac{area(Intersection)}{area(Union)}
$$

На практике IoU используется в нескольких местах:

- при сопоставлении предсказаний и ground truth  
- при решении, считать ли detection true positive  
- при подавлении дубликатов

### Confidence score

Confidence score показывает, насколько уверена модель в предсказании. В реальности финальный score часто включает в себя сразу несколько факторов:

- объектность  
- вероятность класса  
- косвенно качество локализации

### NMS

Non-Maximum Suppression убирает дубликаты. Если несколько боксов сильно перекрываются, мы оставляем самый сильный, а остальные подавляем.

Это важная идея: сырые предсказания детектора почти всегда нуждаются в постобработке, чтобы стать удобными для пользователя.


## 4. Почему детекцию труднее оценивать

В классификации часто хватает accuracy. В детекции этого недостаточно, потому что нас интересует сразу несколько вещей:

- нашла ли модель объект  
- правильно ли определила класс  
- достаточно ли хорошо локализовала объект  
- ставит ли хорошие предсказания выше плохих

Именно поэтому стандартом стали AP и mAP. Они учитывают поведение precision-recall кривой, а не одну произвольно выбранную точку отсечения.




## AP и mAP

`AP` и `mAP` — это основные метрики качества в задаче **детекции объектов**.

Они нужны потому, что в детекции мало просто “угадать класс”. Модель должна одновременно:

- найти объект
- правильно определить его класс
- достаточно точно поставить bounding box

## Что такое AP

`AP` = `Average Precision`.

Это метрика качества **для одного класса**.

Например, можно отдельно посчитать `AP` для класса `person`, отдельно для `car`, отдельно для `dog`.

### Основная идея

Детектор выдает много предсказаний с разными значениями уверенности (`confidence score`).

Если мы будем менять порог уверенности, то будет меняться баланс между:

- `precision` — насколько много найденных объектов действительно правильные
- `recall` — насколько много реальных объектов модель вообще смогла найти

Из этих значений строится **precision-recall curve**.

Тогда `AP` — это площадь под этой кривой:

$$
AP = \int_0^1 Precision(Recall)\, dRecall
$$

Интуитивно:

- высокий `AP` означает, что модель хорошо ранжирует предсказания
- правильные detections получают более высокую уверенность, чем неправильные
- при изменении порога модель сохраняет хороший баланс между precision и recall

## Что такое precision и recall

В детекции сначала нужно определить, какое предсказание считается **правильным**.

Обычно предсказание считается `true positive`, если:

- класс определен верно
- `IoU` с истинным bounding box выше заданного порога, например `0.5`

Тогда:

$$
Precision = \frac{TP}{TP + FP}
$$

$$
Recall = \frac{TP}{TP + FN}
$$

Где:

- `TP` — правильно найденные объекты
- `FP` — ложные срабатывания
- `FN` — пропущенные объекты

## Почему AP лучше, чем одна accuracy

Обычная `accuracy` плохо подходит для детекции, потому что здесь:

- на одном изображении может быть много объектов
- модель может выдать много предсказаний
- важен порядок предсказаний по confidence
- важна точность локализации через `IoU`

`AP` гораздо лучше отражает реальное качество детектора.

## Что такое mAP

`mAP` = `mean Average Precision`.

Это **среднее значение AP по нескольким классам**.

Если у нас `K` классов, то:

$$
mAP = \frac{1}{K}\sum_{k=1}^{K} AP_k
$$

То есть:

- сначала считаем `AP` для каждого класса отдельно
- потом усредняем результаты

Пример:

- `AP(person)=0.82`
- `AP(car)=0.76`
- `AP(dog)=0.71`

Тогда:

$$
mAP = \frac{0.82 + 0.76 + 0.71}{3} = 0.763
$$

## Важное уточнение про IoU threshold

Когда пишут `mAP@0.5`, это означает:

- предсказание считается правильным, если `IoU >= 0.5`

Когда пишут `mAP@0.5:0.95`, это означает:

- метрика усредняется по нескольким порогам `IoU`
- обычно от `0.50` до `0.95` с шагом `0.05`

Это более строгая метрика.

### На практике

- `mAP@0.5` обычно выше и “мягче”
- `mAP@0.5:0.95` строже и лучше показывает реальную точность локализации

## Как интерпретировать эти метрики

- высокий `AP` для класса `person` означает, что модель хорошо находит людей
- высокий `mAP` означает, что модель в среднем хорошо работает по всем классам
- если `mAP@0.5` высокий, а `mAP@0.5:0.95` заметно ниже, то модель находит объекты, но ставит bounding boxes не очень точно

## Короткая интуиция

- `AP` — качество детекции **для одного класса**
- `mAP` — среднее качество детекции **по всем классам**

Если совсем коротко:

> `AP` показывает, насколько хорошо детектор работает для одного класса,  
> а `mAP` показывает, насколько хорошо он работает в среднем по всем классам.



## 5. One-stage и two-stage детекторы

### Two-stage

Примеры: Faster R-CNN, Mask R-CNN

Идея:

1. сначала предложить regions of interest  
2. потом классифицировать и уточнить их

Такие модели часто очень точны и логичны с архитектурной точки зрения, но обычно медленнее.

### One-stage

Примеры: SSD, RetinaNet, YOLO

Они сразу предсказывают боксы и классы за один проход. Это делает их быстрее и удобнее для real-time сценариев.

В этом ноутбуке мы разбираем **Faster R-CNN**, потому что это особенно удачная архитектура для обучения.


In [3]:
def box_iou_xyxy(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter = inter_w * inter_h

    area_a = max(0, xa2 - xa1) * max(0, ya2 - ya1)
    area_b = max(0, xb2 - xb1) * max(0, yb2 - yb1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


box_a = (40, 30, 150, 140)
box_b = (80, 60, 180, 170)

print("IoU =", round(box_iou_xyxy(box_a, box_b), 3))

if MPL_AVAILABLE:
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.set_xlim(0, 220)
    ax.set_ylim(200, 0)
    ax.add_patch(patches.Rectangle((40, 30), 110, 110, fill=False, linewidth=3, edgecolor="tab:blue"))
    ax.add_patch(patches.Rectangle((80, 60), 100, 110, fill=False, linewidth=3, edgecolor="tab:orange"))
    ax.set_title("Два прямоугольника и их пересечение")
    plt.show()


IoU = 0.32


AttributeError: module 'PIL.Image' has no attribute 'Resampling'

<Figure size 288x288 with 1 Axes>

![Faster R-CNN](images/faster_rcnn_architecture.svg)



In [4]:
if not TORCH_AVAILABLE:
    print("Для пайплайна детекции нужен PyTorch.")
else:
    from torchvision.datasets import PennFudanPed
    from torchvision.transforms import v2 as T

    DATA_ROOT = Path.cwd() / "data" / "pennfudan"

    image_tf = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

    class PennFudanDetectionDataset(Dataset):
        def __init__(self, root: Path):
            self.ds = PennFudanPed(root=root, download=True)

        def __len__(self):
            return len(self.ds)

        def __getitem__(self, idx):
            image, target = self.ds[idx]
            image = image_tf(image)
            boxes = target["boxes"].float()
            labels = target["labels"].long()
            return image, {"boxes": boxes, "labels": labels}

    det_ds = PennFudanDetectionDataset(DATA_ROOT)
    print("Penn-Fudan samples:", len(det_ds))


ImportError: cannot import name 'OpOverload' from 'torch._ops' (/Users/hiber/miniforge3/envs/venv/lib/python3.8/site-packages/torch/_ops.py)

## 6. Faster R-CNN по этапам

### Backbone

CNN извлекает feature map из изображения. Это общее представление, на котором работают последующие блоки.

### Region Proposal Network (RPN)

RPN сканирует feature map и предлагает области, которые, скорее всего, содержат объекты. Это еще не финальные detection, а candidate regions.

### ROI stage

Предложенные области преобразуются в фиксированный размер признакового представления. Так следующий head может обрабатывать их единообразно.

### Detection head

Для каждого proposal head предсказывает:

- class scores  
- уточненные координаты бокса

Именно логика «сначала proposal, потом классификация и refinement» делает Faster R-CNN two-stage детектором.


## 7. Anchors и proposals

Anchor — это заранее заданная форма candidate box, привязанная к определенной позиции на feature map.

Зачем это нужно?

Потому что детектору трудно искать произвольные боксы в непрерывном пространстве с нуля. Anchors дают ему разумные стартовые гипотезы для разных масштабов и соотношений сторон.

Даже если современная модель использует другую стратегию назначения, общая интуиция остается важной:

**детекции нужен систематический способ связать позиции на изображении с возможными прямоугольниками.**


In [ ]:
if not TORCH_AVAILABLE:
    print("Пропуск визуализации детекции: PyTorch недоступен.")
else:
    image, target = det_ds[0]
    print("Image shape:", tuple(image.shape))
    print("Boxes shape:", tuple(target["boxes"].shape))

    if MPL_AVAILABLE:
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.imshow(image.permute(1, 2, 0))
        for box in target["boxes"]:
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(
                patches.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    linewidth=2,
                    edgecolor="red",
                )
            )
        ax.set_title("Пример Penn-Fudan с истинными боксами")
        ax.axis("off")
        plt.show()


In [ ]:
if not TORCH_AVAILABLE:
    print("Пропуск создания детектора: PyTorch недоступен.")
else:
    from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, fasterrcnn_resnet50_fpn
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

    detector = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    in_features = detector.roi_heads.box_predictor.cls_score.in_features
    detector.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)

    total_params = sum(p.numel() for p in detector.parameters())
    print("Detector parameters:", f"{total_params:,}")


## 8. Реальный датасет: Penn-Fudan Pedestrian

Для практики используется Penn-Fudan Pedestrian — тот же небольшой реальный датасет, что и в официальном tutorial `torchvision`.

Почему он хорош для обучения:

- реальные уличные сцены  
- простая задача одного класса  
- легко интерпретировать результаты визуально  
- небольшой размер для быстрых экспериментов

Важно заметить отличие от сегментации: target — это уже не плотная маска, а список объектов переменной длины. Из-за этого меняется и dataloader, и `collate_fn`.


In [ ]:
if not TORCH_AVAILABLE:
    print("Пропуск шага обучения: PyTorch недоступен.")
else:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    detector = detector.to(DEVICE)

    def collate_fn(batch):
        images, targets = zip(*batch)
        return list(images), list(targets)

    train_loader = DataLoader(
        torch.utils.data.Subset(det_ds, list(range(min(16, len(det_ds))))),
        batch_size=2,
        shuffle=True,
        collate_fn=collate_fn,
    )

    optimizer = torch.optim.Adam([p for p in detector.parameters() if p.requires_grad], lr=1e-4)

    detector.train()
    images, targets = next(iter(train_loader))
    images = [img.to(DEVICE) for img in images]
    targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

    losses = detector(images, targets)
    total_loss = sum(losses.values())
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    print("Потери на одном demo-шаге:")
    for name, value in losses.items():
        print(f"  {name}: {value.item():.4f}")


## 9. Почему в коде используется `fasterrcnn_resnet50_fpn`

Этот выбор одновременно практический и учебный.

Практический, потому что:

- есть pretrained weights  
- модель входит в `torchvision`  
- для fine-tuning достаточно заменить небольшую голову

Учебный, потому что:

- архитектура широко известна в литературе  
- она ясно показывает стандартный detection workflow  
- студент легко видит, где именно находятся классификатор и box regressor


## 10. AP и mAP простыми словами

Average Precision суммирует поведение precision-recall кривой для одного класса. Mean Average Precision усредняет AP по классам.

Почему это лучше, чем один фиксированный threshold?

Потому что детектор — это не только локализатор, но и система ранжирования. Если модель ставит хорошие detections выше плохих, это должно отражаться в метрике, даже до выбора конкретного порога.

Именно поэтому серьезную детекцию почти никогда не оценивают одной “accuracy”-подобной метрикой.


## 11. Где детекция используется на практике

Детекция полезна везде, где достаточно примерной локализации:

- аналитика трафика  
- мониторинг полок в ритейле  
- системы безопасности работников  
- спортивная аналитика  
- робототехника и складские системы  
- мониторинг животных и дроны

Во многих production-системах детекция — это front-end этап. Дальше boxes подаются в tracker, counter, OCR или другой downstream-модуль.


In [ ]:
if not TORCH_AVAILABLE:
    print("Пропуск инференса: PyTorch недоступен.")
else:
    detector.eval()
    demo_image, demo_target = det_ds[1]

    with torch.no_grad():
        predictions = detector([demo_image.to(DEVICE)])[0]

    keep = predictions["scores"].cpu() > 0.5
    pred_boxes = predictions["boxes"].cpu()[keep]
    pred_scores = predictions["scores"].cpu()[keep]

    print("Predicted boxes kept after score threshold:", len(pred_boxes))

    if MPL_AVAILABLE:
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.imshow(demo_image.permute(1, 2, 0))
        for box, score in zip(pred_boxes, pred_scores):
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(
                patches.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    linewidth=2,
                    edgecolor="lime",
                )
            )
            ax.text(x1, y1 - 4, f"{score:.2f}", color="lime", fontsize=10, backgroundcolor="black")
        ax.set_title("Предсказания Faster R-CNN")
        ax.axis("off")
        plt.show()


## 12. Типичные ошибки и режимы отказа

Ошибки новичка:

- неправильный формат боксов  
- отсутствие собственного `collate_fn`  
- оценка только «на глаз»  
- неверная интерпретация threshold как качества модели

Частые ошибки самой модели:

- дублирующиеся боксы  
- пропуск маленьких объектов  
- плохая локализация при перекрытиях  
- смещение боксов на фон

При отладке детектора важно смотреть и на:

- сырые предсказания  
- отфильтрованные предсказания после threshold и NMS


In [ ]:
# Заготовка упражнения: сравните score thresholds

if not TORCH_AVAILABLE:
    print("Для упражнения по детекции нужен PyTorch.")
else:
    for thr in [0.3, 0.5, 0.7]:
        keep = predictions["scores"].cpu() > thr
        print(f"threshold={thr:.1f} -> kept boxes={int(keep.sum())}")


## Упражнения

1. Понизьте score threshold и посмотрите на компромисс между precision и количеством «мусора».  
2. Замените Faster R-CNN на RetinaNet или SSD и сравните поведение.  
3. Переведите несколько боксов вручную между `xyxy`, `xywh` и `cxcywh`.  
4. Сделайте не один, а несколько optimization steps и посмотрите, как меняются предсказания.


## Источники

- Материал расширен на основе материалов курса и [deepmachinelearning.ru](https://deepmachinelearning.ru/docs/Neural-networks/Object-detection).  
- Официальный tutorial `torchvision` по object detection finetuning.  
- Статья Faster R-CNN: *Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks*.
